# To Trade Or Not To Trade: Cascading Waterfall Round Robin Rebalancing Mechanism for Cryptocurrencies

Authors: Ravi Kashyap
Published: 2024-05-17
ArXiv: [https://arxiv.org/abs/2407.12150](https://arxiv.org/abs/2407.12150)

## Strategy Description
This notebook implements the Cascading Waterfall Round Robin Rebalancing Mechanism for cryptocurrencies. The strategy involves rebalancing the portfolio periodically based on the uncertainty in micro-asset level characteristics and macro-aggregate market factors. The algorithm buys assets that drop in price and sells as they soar, only when certain boundaries are crossed to filter out market noise and ensure sound trade execution.


In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

In [ ]:
# Configuration
UNIVERSE = ['AAPL', 'MSFT']
REBALANCING_PERIOD = '1D'
BOUNDARY_THRESHOLD = 0.05

# Hypothesis
"""
We hypothesize that by rebalancing the portfolio daily based on the price movements of assets,
we can capture volatility and generate alpha in the hyper-volatile crypto market.
""

## Phase 2 — Data Download & Feature Computation

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download data
data = yf.download(UNIVERSE, period='1y', interval=REBALANCING_PERIOD)
prices = data['Adj Close']

# Compute returns
returns = prices.pct_change().dropna()

# Cross-sectional normalization
normalized_returns = (returns - returns.mean()) / returns.std()

## Phase 3 — Signal Generation & Portfolio Construction

In [ ]:
# Generate signals
signals = np.where(normalized_returns < -BOUNDARY_THRESHOLD, 1, np.where(normalized_returns > BOUNDARY_THRESHOLD, -1, 0))

# Shift signals forward by 1 period to avoid look-ahead bias
signals = signals.shift(1)

# Position sizing
positions = signals / signals.sum(axis=1)

# Portfolio construction
portfolio = positions * prices
portfolio_value = portfolio.sum(axis=1))

## Phase 4 — Vectorized Backtest

In [ ]:
# Calculate daily returns of the portfolio
portfolio_returns = portfolio_value.pct_change().dropna()

# Calculate cumulative returns
cumulative_returns = (1 + portfolio_returns).cumprod()

# Plot equity curve
import matplotlib.pyplot as plt

plt.plot(cumulative_returns)
plt.title('Equity Curve')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns')
plt.show()

## Phase 5 — Performance Metrics

In [ ]:
from scipy.stats import norm

# Calculate performance metrics
annual_return = portfolio_returns.mean() * 252
annual_volatility = portfolio_returns.std() * np.sqrt(252)
sharpe_ratio = annual_return / annual_volatility
sortino_ratio = annual_return / portfolio_returns[portfolio_returns < 0].std() * np.sqrt(252)
max_drawdown = (cumulative_returns.cummax() - cumulative_returns) / cumulative_returns.cummax().shift(1)
max_drawdown_value = max_drawdown.max()
calmar_ratio = annual_return / max_drawdown_value

print(f'Annual Return: {annual_return:.2%}')
print(f'Annual Volatility: {annual_volatility:.2%}')
print(f'Sharpe Ratio: {sharpe_ratio:.2f}')
print(f'Sortino Ratio: {sortino_ratio:.2f}')
print(f'Max Drawdown: {max_drawdown_value:.2%}')
print(f'Calmar Ratio: {calmar_ratio:.2f}')

# Plot equity curve
plt.plot(cumulative_returns)
plt.title('Equity Curve')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns')
plt.show()

## Phase 6 — Monitoring Stub

In [ ]:
def monitor_portfolio(prices, positions):
    current_value = (prices * positions).sum(axis=1).iloc[-1]
    daily_pnl = current_value - current_value.shift(1)
    print(f'Daily P&L: {daily_pnl:.2f}')
    print('Current Positions:')
    print(positions.iloc[-1])

# Example usage
monitor_portfolio(prices, positions)